In [77]:
import pandas as pd
import numpy as np

# Keep the original untouched, clean only the copy
loans_raw = pd.read_csv(r'C:\Users\Hey HP\Desktop\gowthami_py\Data Visualisation and Analysis\loans.csv')
loans = loans_raw.copy()

In [78]:
#Task 1: Load and look
loans.shape

loans.info()

loans.head()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,notes
0,L-30001,M0150,A Second Spring,ROMANCE,2026-01-06,2026-01-30,"7,50 kr",Yes,NaN
1,L-30002,M0037,The Silent Harbour,Crime,2026-01-05,2026-01-25,"0,00 kr",YES,NaN
2,L-30003,M0143,The Glass Mage,fantasy,2026-01-05,2026-01-26,"0,00 kr",YES,NaN
3,L-30003,M0143,The Glass Mage,fantasy,2026-01-05,2026-01-26,"0,00 kr",YES,NaN
4,L-30004,M0121,Dead Reckoning Lane,CRIME,2026-01-05,2026-01-12,"0,00 kr",Y,NaN


In [79]:
# 623 rows and 9 columns.
# Problems: loan_date, return_date and late_fee are text but should be dates/numbers.
# returned is text but should be True/False.
# notes is completely empty (0 non-null values).
# return_date has only 593 values, so 30 are missing.
# genre probably has messy spelling (spaces, capitals).

In [80]:
#Task 2: Clean the genre

print(loans['genre'].nunique())

loans['genre'] = loans['genre'].str.strip().str.lower()
print(loans['genre'].nunique())
print(loans['genre'].unique())                             

28
9
<StringArray>
[        'romance',           'crime',         'fantasy',        'children',
      'children's',         'history',          'sci-fi', 'science fiction',
          'sci fi']
Length: 9, dtype: str


In [81]:
loans['genre'] = loans['genre'].replace({
    'scifi': 'sci-fi',
    'sci fi': 'sci-fi',
    'childrens': 'children',
})

loans['genre'] = loans['genre'].str.title()       
loans['genre'].nunique()                          
loans['genre'].value_counts()


genre
Crime              163
Fantasy             90
Romance             86
Children            76
Children'S          59
History             59
Sci-Fi              57
Science Fiction     33
Name: count, dtype: int64

In [82]:
#Task 3: Yes and no
print(loans['returned'].nunique())
print(loans['returned'].value_counts())                 

10
returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64


In [83]:
loans['returned'] = (loans['returned']
                     .str.strip()
                     .str.lower()
                     .map({'yes': True, 'y': True, 'no': False, 'n': False}))

print(loans['returned'].isna().sum())   # must be 0
loans['returned'].value_counts()        # True 593, False 30

0


returned
True     593
False     30
Name: count, dtype: int64

In [84]:
loans = loans_raw.copy()

loans['returned'].str.strip().str.lower().value_counts()
loans['returned'] = (loans['returned']
                     .str.strip()
                     .str.lower()
                     .map({'yes': True, 'y': True, 'no': False, 'n': False}))

loans['returned'].isna().sum()        # must be 0

loans['returned'].value_counts()      

returned
True     593
False     30
Name: count, dtype: int64

In [85]:
#Task 4: The late fee is text
loans['late_fee'].head(8).tolist()
#loans['late_fee'].astype(float)

loans['late_fee'] = (loans['late_fee']
                     .str.replace('kr', '')      # 1. remove kr:      '7,50 '
                     .str.replace(' ', '')       # 2. remove spaces:  '7,50'
                     .str.replace(',', '.')      # 3. comma to dot:   '7.50'
                     .astype(float))             # 4. only now: 7.5

In [86]:
# On purpose: astype(float) on the original text. It fails.
# (Here I catch the error so the notebook keeps running. In your own notebook, let it fail and read it.)
try:
    loans['late_fee'].astype(float)
except ValueError as e:
    print('ValueError:', e)

In [87]:

# late fee (this cell can only run ONCE after the fresh copy)
loans['late_fee'] = (loans_raw['late_fee']
                     .str.replace('kr', '')
                     .str.replace(' ', '')
                     .str.replace(',', '.')
                     .astype(float))
print(loans['late_fee'].describe().round(2))
print('Total fees:', loans['late_fee'].sum())     # 7677.5

count    623.00
mean      12.32
std       16.11
min        0.00
25%        0.00
50%        0.00
75%       25.00
max       47.50
Name: late_fee, dtype: float64
Total fees: 7677.5


In [88]:
#Task 5: Dates
loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

loans['loan_date'].isna().sum()                   
loans['loan_date'].min()                          
loans['loan_date'].max()                          

per_month = loans['loan_date'].dt.to_period('M').value_counts().sort_index()
per_month                                         
per_month.sum()                                   

np.int64(620)

In [89]:
#Task 6: Duplicates
loans_raw.duplicated().sum()                      
loans.duplicated().sum()                          
loans.duplicated(subset='loan_id').sum()          

np.int64(18)

In [90]:
total_before = loans['late_fee'].sum()            

loans = loans.drop_duplicates().reset_index(drop=True)
loans.shape                                       

total_after = loans['late_fee'].sum()             
diff = total_before - total_after                 
diff / total_before * 100                         

np.float64(1.7909475740801044)

In [91]:
#Task 6: Duplicates
loans_raw.duplicated().sum()                      # 12
loans.duplicated().sum()                          # 18
loans.duplicated(subset='loan_id').sum()          # 18

np.int64(6)

In [92]:
total_before = loans['late_fee'].sum()            # 7677.5

loans = loans.drop_duplicates().reset_index(drop=True)
loans.shape                                       # (605, 9)

total_after = loans['late_fee'].sum()             # 7447.5
diff = total_before - total_after                 # 230.0
diff / total_before * 100                         # about 3.0 %

np.float64(0.0)

In [5]:
## Task 7: Put it together
# One cell that starts from loans_raw every time, so it works again and again.
loans = loans_raw.copy()

for run in [1, 2]:
    loans = loans_raw.copy()# I run it twice to test
    loans = loans.drop(columns='notes')  # empty column

    loans['genre'] = (loans['genre'].str.strip().str.lower()
                      .replace({"children's": 'children', 'science fiction': 'sci-fi', 'sci fi': 'sci-fi'})
                      .str.title())

    loans['returned'] = (loans['returned'].str.strip().str.lower()
                         .map({'yes': True, 'y': True, 'no': False, 'n': False}))

    loans['late_fee'] = (loans['late_fee'].str.replace('kr', '').str.replace(' ', '')
                         .str.replace(',', '.').astype(float))

    loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

    loans = loans.drop_duplicates().reset_index(drop=True)   # duplicates last
    print('Run', run, loans.shape)

loans.to_csv('data/loans_clean.csv', index=False)



NameError: name 'loans_raw' is not defined